This notebook is to play around with gui code for visualizing data online

In [42]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any




In [11]:
# Load config
import sys
import os
from pathlib import Path

# 

# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))

home directory: /gpfs01/euler/User/ssuhai
Config path: GitRepos/simulation_closed_loop/config
Configuration loaded successfully:
data_subfolders:
  day: 20250717
  experiment: 1
DJ:
  username: ssuhai
  userinfo:
    experimenter: closedlooptest
    animal_loc: 1
    region_loc: 2
    field_loc: 3
    stimulus_loc: 4
    cond1_loc: 5
    data_dir: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment
  table_parameters:
    PreprocessParams: {}
    Stimulus:
      noise:
        stim_name: densenoise
        stim_family: noise
        pix_n_x: 20
        pix_n_y: 15
        skip_duplicates: true
        pix_scale_x_um: 40
        pix_scale_y_um: 40
        framerate: 5
    DNoiseTraceParams:
      dnoise_params_id: 1
      fupsample_trace: 20
      fupsample_stim: 4
      ref_time: stim
      fit_kind: gradient
      skip_duplicates: true
      pre_blur_sigma_s: 0.0
      post_blur_sigma_s: 0.0
paths:
  repo_directory: /gpfs01/euler/User

In [13]:
# 
from simulations.loop_components.dj_wrappers import OpenRetinaWrapper
from simulations.loop_components.stimulus_file_copier import copy_stim_files,create_directory_structure
from simulations.loop_components.model_to_stimulus import from_data_to_mei_video

# connect populated closed loop schema

In [14]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

openretinawrapper = OpenRetinaWrapper(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )

In [15]:

# Load config and tables
openretinawrapper.load_config()
openretinawrapper.load_tables()
print("OpenRetinaWrapper loaded and configured successfully")

[2025-07-21 15:44:38,930][INFO]: Connecting ssuhai@172.25.240.200:3306
[2025-07-21 15:44:38,990][INFO]: Connected ssuhai@172.25.240.200:3306
[2025-07-21 15:44:38,990][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
OpenRetinaWrapper loaded and configured successfully
OpenRetinaWrapper loaded and configured successfully


# GUI Components for Visualization

Let's create a basic GUI to visualize the data processed by the OpenRetinaWrapper.

In [71]:
from simulations.gui.data_visualization import VisualizationGUI

In [49]:
np.array([1,2]).item()

ValueError: can only convert an array of size 1 to a Python scalar

In [56]:
(openretinawrapper('CelltypeAssignment')() & dict(roi_id=1))

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),avg_stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,os_ds_stim_name Unique string identifier,training_data_hash hash of the classifier training data files,classifier_params_hash hash of the classifier params config,"celltype predicted group, without quality or confidence threshold",max_confidence confidence score for assigned celltype for easy restriction,confidence confidence score (probability) for all celltypes
closedlooptest,2025-07-17,1,1,GCL1,RR,iter0,gChirp,control,1,1,movingbar,7c5bdca5b59dd970f8e1aebfe86323db,e705a7d0cb4119f76d9064a57a2b527f,27,0.245469,=BLOB=


In [67]:
celltype_data = (openretinawrapper('CelltypeAssignment')() & dict(roi_id=1)).fetch1()
confidence_scores = celltype_data['confidence']

# Get top 3 with scores
top_3_indices = confidence_scores.argsort()[-3:][::-1]
top_3_groups = top_3_indices + 1 # assume index based group assignment
top_3_scores = confidence_scores[top_3_indices]

assert celltype_data["celltype"] == top_3_groups[0], "Top group does not match celltype assignment"

In [64]:
top_3_indices

array([26, 16, 30])

In [63]:
top_3_scores

array([0.24546905, 0.23105055, 0.12667192])

In [66]:
(openretinawrapper('CelltypeAssignment')() & dict(roi_id=1))

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),avg_stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,os_ds_stim_name Unique string identifier,training_data_hash hash of the classifier training data files,classifier_params_hash hash of the classifier params config,"celltype predicted group, without quality or confidence threshold",max_confidence confidence score for assigned celltype for easy restriction,confidence confidence score (probability) for all celltypes
closedlooptest,2025-07-17,1,1,GCL1,RR,iter0,gChirp,control,1,1,movingbar,7c5bdca5b59dd970f8e1aebfe86323db,e705a7d0cb4119f76d9064a57a2b527f,27,0.245469,=BLOB=


In [44]:
openretinawrapper.tables

{'UserInfo': djimaging.schemas.core_schema.UserInfo,
 'Experiment': djimaging.schemas.core_schema.Experiment,
 'Field': djimaging.schemas.core_schema.Field,
 'OpticDisk': djimaging.schemas.full_rgc_schema.OpticDisk,
 'RelativeFieldLocation': djimaging.schemas.full_rgc_schema.RelativeFieldLocation,
 'RelativeRoiLocation': djimaging.user.ssuhai.schemas.ssuhai_schema_closed_loop.RelativeRoiLocation,
 'Stimulus': djimaging.schemas.core_schema.Stimulus,
 'RoiMask': djimaging.schemas.core_schema.RoiMask,
 'Roi': djimaging.schemas.core_schema.Roi,
 'Traces': djimaging.schemas.core_schema.Traces,
 'Presentation': djimaging.schemas.core_schema.Presentation,
 'RawDataParams': djimaging.schemas.core_schema.RawDataParams,
 'PreprocessParams': djimaging.schemas.core_schema.PreprocessParams,
 'PreprocessTraces': djimaging.schemas.core_schema.PreprocessTraces,
 'Snippets': djimaging.schemas.core_schema.Snippets,
 'Averages': djimaging.schemas.core_schema.Averages,
 'ChirpQI': djimaging.schemas.full_r

In [75]:
# Import numpy which was missing from the above code
import numpy as np

# Create the visualization GUI instance
visualization_gui = VisualizationGUI(openretinawrapper)

# Create and display the GUI
gui_widget = visualization_gui.start_gui()
display(gui_widget)



Loaded 89 ROIs
